# Self-Improving Agents with Structured Reflection

## Introduction

Most agent workflows treat prompts as static — you write a system prompt, deploy it, and hope it works. But what if your agent could evaluate its own performance across multiple tasks, identify its weakest area, and rewrite its own instructions to improve?

This cookbook implements a **self-improvement cycle**: a structured loop where an agent runs a batch of tasks, a reflection step evaluates performance across dimensions, and then the agent's prompt is automatically revised to target the weakest area. The result is measurable improvement without manual prompt tuning.

### How This Differs from Evaluator-Optimizer

The [evaluator-optimizer pattern](evaluator_optimizer.ipynb) iterates on a **single output** until it passes. This pattern improves the **agent itself** across a batch of tasks by modifying its underlying prompt. Think of it as the difference between proofreading one essay vs. improving your writing ability.

| | Evaluator-Optimizer | Self-Improving Agent |
|-|---|---|
| **What improves** | One specific output | The agent's prompt/behavior |
| **Feedback scope** | Single task | Batch of tasks |
| **Persistence** | Output is refined, prompt unchanged | Prompt is revised for future tasks |
| **Goal** | Get this answer right | Get all future answers better |

### What You'll Build

A customer support agent that:
1. Responds to a batch of customer emails
2. Gets scored across three dimensions: **empathy**, **accuracy**, and **actionability**
3. Reflects on its weakest dimension
4. Rewrites its own system prompt to target that weakness
5. Demonstrates measurable improvement on the next batch

### Prerequisites

- Python 3.9+
- Anthropic API key: `export ANTHROPIC_API_KEY='your-key'`
- Basic understanding of prompt engineering

### When to Use This Pattern

**Use this pattern when:**
- Your agent handles diverse tasks where no single prompt is optimal
- You have clear evaluation criteria you can score against
- You want automated prompt improvement without manual iteration
- You need an audit trail of what changed and why

**Don't use this pattern when:**
- You need a one-shot answer (use evaluator-optimizer instead)
- Evaluation criteria are subjective or hard to define
- The task space is narrow enough that manual prompt tuning suffices

## Setup

### Helper Functions

This cookbook uses the shared `util.py` helpers:
- `llm_call(prompt, system_prompt, model)`: Sends a prompt to Claude and returns text
- `extract_xml(text, tag)`: Extracts content from XML tags

See [util.py](util.py) for the implementation.

In [ ]:
import json
import re

from util import extract_xml, llm_call

MODEL = "claude-sonnet-4-6"

## The Self-Improvement Cycle

The cycle has four phases:

1. **Execute** — Run the agent against a batch of tasks using the current prompt
2. **Evaluate** — Score each response across defined dimensions
3. **Reflect** — Analyze scores to identify the weakest dimension and root cause
4. **Revise** — Rewrite the prompt to specifically target the identified weakness

```
┌──────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐
│ Execute  │ ──▶ │ Evaluate │ ──▶ │ Reflect  │ ──▶ │  Revise  │
│ (batch)  │     │ (score)  │     │ (analyze)│     │ (prompt) │
└──────────┘     └──────────┘     └──────────┘     └────┬─────┘
     ▲                                                   │
     └───────────────────────────────────────────────────┘
```

Each cycle produces a revised prompt and a record of what changed and why — an audit trail that makes the improvement process transparent and reversible.

## Implementation

### Test Scenarios

First, we define a batch of customer emails that represent the range of tasks our agent needs to handle. These stay constant across improvement cycles so we can measure progress.

In [ ]:
TEST_EMAILS = [
    {
        "id": "billing_frustration",
        "email": (
            "I've been charged twice for my subscription this month and nobody "
            "has responded to my last THREE support tickets. This is unacceptable. "
            "I want a refund immediately or I'm disputing with my bank."
        ),
    },
    {
        "id": "feature_request",
        "email": (
            "Love the product! Any chance you could add dark mode? I use it "
            "late at night and the bright screen is tough on my eyes. Would "
            "happily pay more for a premium tier with this feature."
        ),
    },
    {
        "id": "technical_issue",
        "email": (
            "The export function keeps failing with a timeout error when I try "
            "to export more than 500 rows. I've tried Chrome and Firefox. This "
            "is blocking my end-of-quarter report that's due tomorrow."
        ),
    },
    {
        "id": "cancellation",
        "email": (
            "I need to cancel my account. I've switched to a competitor because "
            "your API rate limits are too low for our use case. Is there a way "
            "to export my data before the account is closed?"
        ),
    },
]

### Phase 1: Execute

Run the agent against each test email using the current system prompt.

In [ ]:
def execute_batch(system_prompt: str, test_cases: list[dict]) -> list[dict]:
    """Run the agent against a batch of test cases."""
    results = []
    for case in test_cases:
        response = llm_call(
            prompt=f"Customer email:\n{case['email']}\n\nWrite your response:",
            system_prompt=system_prompt,
            model=MODEL,
        )
        results.append({
            "id": case["id"],
            "email": case["email"],
            "response": response,
        })
    return results

### Phase 2: Evaluate

Score each response across three dimensions on a 1-5 scale. Using a separate LLM call for evaluation ensures the agent isn't grading its own work with built-in bias.

In [ ]:
EVAL_PROMPT = """
You are evaluating a customer support agent's response.

Customer email:
{email}

Agent response:
{response}

Score the response on three dimensions (1-5 each):

- **Empathy**: Does the response acknowledge the customer's feelings and situation?
  1=ignores emotions, 3=acknowledges but generic, 5=deeply personalized understanding

- **Accuracy**: Is the response factually appropriate and does it avoid making promises
  the agent can't keep? 1=wrong/misleading, 3=correct but vague, 5=precise and appropriate

- **Actionability**: Does the response give the customer clear next steps?
  1=no guidance, 3=some direction, 5=specific steps with expected outcomes

Return scores and reasoning in this format:

<scores>
<empathy>score</empathy>
<accuracy>score</accuracy>
<actionability>score</actionability>
</scores>
<reasoning>Brief explanation of each score</reasoning>
"""


def evaluate_batch(results: list[dict]) -> list[dict]:
    """Evaluate each response across defined dimensions."""
    evaluated = []
    for result in results:
        eval_response = llm_call(
            prompt=EVAL_PROMPT.format(
                email=result["email"],
                response=result["response"],
            ),
            model=MODEL,
        )
        scores_xml = extract_xml(eval_response, "scores")
        evaluated.append({
            **result,
            "scores": {
                "empathy": int(extract_xml(scores_xml, "empathy").strip()),
                "accuracy": int(extract_xml(scores_xml, "accuracy").strip()),
                "actionability": int(extract_xml(scores_xml, "actionability").strip()),
            },
            "reasoning": extract_xml(eval_response, "reasoning").strip(),
        })
    return evaluated

### Phase 3: Reflect

This is the key differentiator from simple eval loops. Instead of just checking pass/fail, reflection analyzes patterns **across the entire batch** to identify the systemic weakness and its root cause.

In [ ]:
REFLECT_PROMPT = """
You are analyzing a batch of customer support agent evaluations to identify
the single most impactful improvement to the agent's system prompt.

Current system prompt:
<current_prompt>
{system_prompt}
</current_prompt>

Evaluation results:
{eval_summary}

Average scores:
- Empathy: {avg_empathy:.1f}/5
- Accuracy: {avg_accuracy:.1f}/5
- Actionability: {avg_actionability:.1f}/5

Identify the weakest dimension and analyze WHY the current prompt produces
weak results in that area. Be specific about what's missing or misleading
in the prompt.

<weakest_dimension>name of the weakest dimension</weakest_dimension>
<root_cause>specific analysis of why the prompt fails here</root_cause>
<improvement>concrete change to make to the prompt</improvement>
"""


def reflect(system_prompt: str, evaluated: list[dict]) -> dict:
    """Analyze evaluation results to identify the highest-impact improvement."""
    # Build evaluation summary
    eval_summary = ""
    for result in evaluated:
        eval_summary += (
            f"\nCase: {result['id']}\n"
            f"  Scores: empathy={result['scores']['empathy']}, "
            f"accuracy={result['scores']['accuracy']}, "
            f"actionability={result['scores']['actionability']}\n"
            f"  Reasoning: {result['reasoning']}\n"
        )

    # Calculate averages
    dimensions = ["empathy", "accuracy", "actionability"]
    averages = {
        dim: sum(r["scores"][dim] for r in evaluated) / len(evaluated)
        for dim in dimensions
    }

    response = llm_call(
        prompt=REFLECT_PROMPT.format(
            system_prompt=system_prompt,
            eval_summary=eval_summary,
            avg_empathy=averages["empathy"],
            avg_accuracy=averages["accuracy"],
            avg_actionability=averages["actionability"],
        ),
        model=MODEL,
    )

    return {
        "averages": averages,
        "weakest_dimension": extract_xml(response, "weakest_dimension").strip(),
        "root_cause": extract_xml(response, "root_cause").strip(),
        "improvement": extract_xml(response, "improvement").strip(),
    }

### Phase 4: Revise

The revision step rewrites the system prompt to address the identified weakness. It receives the current prompt, the reflection analysis, and produces a new prompt that specifically targets the root cause while preserving what already works.

In [ ]:
REVISE_PROMPT = """
You are rewriting a customer support agent's system prompt to improve its
performance. The agent was evaluated across a batch of customer emails and
the weakest dimension was identified.

Current system prompt:
<current_prompt>
{system_prompt}
</current_prompt>

Weakest dimension: {weakest_dimension}
Root cause: {root_cause}
Suggested improvement: {improvement}

Rewrite the system prompt to specifically address this weakness.
Rules:
- Keep everything that already works well
- Add specific instructions that target the root cause
- Do not make the prompt more than 50% longer than the original
- Be concrete, not vague ("acknowledge the customer's specific situation"
  is better than "be more empathetic")

<revised_prompt>your rewritten system prompt here</revised_prompt>
"""


def revise_prompt(system_prompt: str, reflection: dict) -> str:
    """Rewrite the system prompt to address the identified weakness."""
    response = llm_call(
        prompt=REVISE_PROMPT.format(
            system_prompt=system_prompt,
            weakest_dimension=reflection["weakest_dimension"],
            root_cause=reflection["root_cause"],
            improvement=reflection["improvement"],
        ),
        model=MODEL,
    )
    return extract_xml(response, "revised_prompt").strip()

### Putting It Together: The Improvement Loop

Now we wire the four phases into a single loop that runs for a specified number of cycles. Each cycle produces a log entry with scores, the identified weakness, and the prompt revision.

In [ ]:
def run_improvement_cycle(
    initial_prompt: str,
    test_cases: list[dict],
    num_cycles: int = 2,
) -> list[dict]:
    """Run the full self-improvement loop."""
    current_prompt = initial_prompt
    history = []

    for cycle in range(num_cycles):
        print(f"\n{'=' * 80}")
        print(f"CYCLE {cycle + 1}")
        print(f"{'=' * 80}")

        # Phase 1: Execute
        print("\n[Execute] Running agent against test batch...")
        results = execute_batch(current_prompt, test_cases)

        # Phase 2: Evaluate
        print("[Evaluate] Scoring responses...")
        evaluated = evaluate_batch(results)

        # Phase 3: Reflect
        print("[Reflect] Analyzing patterns...")
        reflection = reflect(current_prompt, evaluated)

        print(f"\n  Scores: ", end="")
        for dim, avg in reflection["averages"].items():
            print(f"{dim}={avg:.1f}", end="  ")
        print(f"\n  Weakest: {reflection['weakest_dimension']}")
        print(f"  Root cause: {reflection['root_cause'][:200]}...")

        # Phase 4: Revise
        print("\n[Revise] Rewriting prompt...")
        new_prompt = revise_prompt(current_prompt, reflection)

        # Log this cycle
        history.append({
            "cycle": cycle + 1,
            "prompt": current_prompt,
            "scores": reflection["averages"],
            "weakest_dimension": reflection["weakest_dimension"],
            "root_cause": reflection["root_cause"],
            "improvement": reflection["improvement"],
            "revised_prompt": new_prompt,
            "per_case_scores": [
                {"id": r["id"], "scores": r["scores"]} for r in evaluated
            ],
        })

        current_prompt = new_prompt

    # Final evaluation with the last revised prompt
    print(f"\n{'=' * 80}")
    print("FINAL EVALUATION")
    print(f"{'=' * 80}")
    print("\n[Execute] Running agent with improved prompt...")
    final_results = execute_batch(current_prompt, test_cases)
    print("[Evaluate] Scoring final responses...")
    final_evaluated = evaluate_batch(final_results)

    dimensions = ["empathy", "accuracy", "actionability"]
    final_averages = {
        dim: sum(r["scores"][dim] for r in final_evaluated) / len(final_evaluated)
        for dim in dimensions
    }

    history.append({
        "cycle": "final",
        "prompt": current_prompt,
        "scores": final_averages,
        "per_case_scores": [
            {"id": r["id"], "scores": r["scores"]} for r in final_evaluated
        ],
    })

    return history

## Running the Improvement Cycle

Let's start with a deliberately minimal system prompt and see how the agent improves itself over two cycles.

In [ ]:
INITIAL_PROMPT = "You are a customer support agent. Respond to customer emails."

history = run_improvement_cycle(
    initial_prompt=INITIAL_PROMPT,
    test_cases=TEST_EMAILS,
    num_cycles=2,
)

## Analyzing the Results

Let's look at how scores changed across cycles and what prompt revisions were made.

In [ ]:
print("Score Progression")
print(f"{'Cycle':<10} {'Empathy':<12} {'Accuracy':<12} {'Actionability':<15} {'Average':<10}")
print("-" * 59)

for entry in history:
    scores = entry["scores"]
    avg = sum(scores.values()) / len(scores)
    cycle_label = str(entry["cycle"])
    print(
        f"{cycle_label:<10} {scores['empathy']:<12.1f} {scores['accuracy']:<12.1f} "
        f"{scores['actionability']:<15.1f} {avg:<10.1f}"
    )

In [ ]:
print("\nImprovement Log")
print("=" * 80)
for entry in history:
    if entry["cycle"] == "final":
        continue
    print(f"\nCycle {entry['cycle']}:")
    print(f"  Weakest dimension: {entry['weakest_dimension']}")
    print(f"  Root cause: {entry['root_cause'][:300]}")
    print(f"  Improvement: {entry['improvement'][:300]}")

print("\n" + "=" * 80)
print("\nFinal Prompt:")
print("-" * 80)
print(history[-1]["prompt"])

## Key Design Decisions

**Why batch evaluation instead of per-task?** Individual evaluations miss systemic patterns. An agent might handle angry customers well but consistently fail on technical issues — you only see this across a batch.

**Why identify only one weakness per cycle?** Changing too many things at once makes it impossible to attribute improvement. Single-variable changes create a clear cause-and-effect trail.

**Why constrain prompt growth?** Without a length constraint, the revision step tends to add instructions without removing anything, eventually creating an unwieldy prompt. The 50% growth limit forces the reviser to integrate improvements into existing instructions.

**Why separate evaluator from agent?** Using the same model for both creates bias — the agent tends to rate its own style highly. A separate evaluation call provides more honest scoring.

## Extending This Pattern

### Adding More Dimensions

The evaluation dimensions are defined in the `EVAL_PROMPT`. Add domain-specific dimensions by extending the scoring criteria:

```python
# For a coding assistant:
dimensions = ["correctness", "readability", "efficiency", "test_coverage"]

# For a sales agent:
dimensions = ["rapport", "qualification", "objection_handling", "next_steps"]
```

### Adding Guardrails

In production, add constraints to the revision step:

- **Rollback checks**: After revision, re-evaluate to ensure previously strong dimensions didn't regress
- **Human approval**: Flag revised prompts for review before deploying
- **Improvement threshold**: Only accept a revision if the weakest dimension improves by at least 0.5 points

### Persisting Improvements

The `history` list provides a full audit trail. In production, persist this to track how your agent's prompt evolves over time:

```python
# Save the improvement history
with open("improvement_log.json", "w") as f:
    json.dump(history, f, indent=2)
```

## Summary

This cookbook demonstrated a self-improvement cycle that turns static prompts into evolving ones. The key insight is that **structured reflection across a batch of tasks** reveals systemic weaknesses that single-task evaluation misses.

The pattern is framework-agnostic — it works with any LLM call interface and any evaluation criteria you define. The four phases (execute, evaluate, reflect, revise) can be adapted to any agent domain where you can define measurable quality dimensions.

### Key Takeaways

- **Batch evaluation reveals patterns** that per-task evaluation misses
- **Single-variable changes** per cycle create traceable improvement
- **Structured reflection** (identifying root cause, not just symptoms) produces targeted revisions
- **Audit trails** make the improvement process transparent and reversible

### Limitations

- **LLM-as-judge variance**: Scores can fluctuate between runs. Mitigate by averaging across larger batches or multiple evaluation runs.
- **Local optima**: The greedy single-dimension strategy may miss improvements that require simultaneous changes across dimensions.
- **Cost**: Each cycle requires `2N + 2` LLM calls (N executions + N evaluations + 1 reflection + 1 revision). Use smaller models for evaluation to reduce cost.